# Notebook 07: DPO -- From RLHF to Direct Preference Optimization

**Sprint 2 | Alignment Track**

---

**What you will learn**:
- The mathematical derivation connecting RLHF and DPO
- How to implement the DPO loss from scratch
- How to build a DPO trainer from scratch and train a model on preference data
- How to use TRL's `DPOTrainer` for production workflows
- When DPO fails and why RLHF may still be preferable

**Prerequisites**: Notebook 06 (RLHF from scratch), familiarity with RL fine-tuning of LLMs.

**Runtime**: Google Colab with T4 GPU (free tier works for small models).

---
## 1. Self-Quiz: Active Recall Before Learning

Before reading anything, answer these in your head (or on paper). Be honest about what you don't know.

1. **What problem does DPO solve compared to RLHF?** What are the practical pain points of RLHF that motivated DPO?
2. **Can you skip the reward model entirely?** If so, how? What replaces it?
3. **What is the beta parameter in DPO?** What happens when it is very large? Very small?
4. **What is the reference model?** Why do we need it? What would happen without it?
5. **Can you write down the DPO loss from memory?** Even approximately?

---
*Come back to these questions after completing this notebook. You should be able to answer all five precisely.*

---
## 2. Setup

In [ ]:
# Install dependencies (Colab-compatible)
!pip install -q torch transformers datasets trl accelerate peft bitsandbytes matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np
import copy
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## 3. The Key Insight: Why DPO Exists

### The RLHF Pipeline is Complex

RLHF requires **3 phases** and **4 models** in memory:

| Phase | Models Needed | What Happens |
|-------|--------------|-------------|
| 1. SFT | SFT model | Supervised fine-tuning on demonstrations |
| 2. Reward modeling | Reward model (initialized from SFT) | Train on human preference pairs |
| 3. RL fine-tuning | Policy + Reference + Reward + Value head | PPO optimization |

In Phase 3, you need **4 models simultaneously**:
- **Policy model** (being trained)
- **Reference model** (frozen copy of SFT model, for KL penalty)
- **Reward model** (frozen, scores outputs)
- **Value head** (critic for PPO advantage estimation)

This is expensive, unstable, and hard to tune.

### The DPO Question

**Can we go directly from preferences to policy, skipping the reward model entirely?**

The answer is **yes**, and the key is a closed-form solution: the optimal policy under the RLHF objective can be expressed analytically in terms of the reward function. By inverting this relationship, we can express the reward in terms of the policy, and substitute directly into the preference model.

In [ ]:
# Let's see this concretely before diving into the math.
# The key relationship: reward = beta * log(pi/pi_ref) + constant

# Simulating what this means:
beta = 0.1

# Suppose we have a policy that assigns different probabilities than the reference
# to a particular response y given prompt x
pi_y_given_x = 0.3       # Policy probability of response y
pi_ref_y_given_x = 0.1   # Reference probability of response y

# The implicit reward for this response:
implicit_reward = beta * np.log(pi_y_given_x / pi_ref_y_given_x)
print(f"Policy increased probability from {pi_ref_y_given_x} to {pi_y_given_x}")
print(f"Implicit reward: {implicit_reward:.4f}")
print()

# If the policy decreased probability:
pi_y_given_x_bad = 0.05
implicit_reward_bad = beta * np.log(pi_y_given_x_bad / pi_ref_y_given_x)
print(f"Policy decreased probability from {pi_ref_y_given_x} to {pi_y_given_x_bad}")
print(f"Implicit reward: {implicit_reward_bad:.4f}")
print()
print("Insight: The log-ratio pi/pi_ref IS the implicit reward (up to a constant).")
print("DPO uses this to train directly on preferences without ever learning an explicit reward model.")

### Why does this work?

Think about it this way: if the optimal policy upweights a response relative to the reference, that response must have high reward. If it downweights a response, that response must have low reward. The *amount* of upweighting directly tells you the reward. So we never need to learn the reward explicitly -- the policy *is* the reward model.

---
## 4. Step-by-Step Mathematical Derivation

This is the core intellectual content for interview preparation. You should be able to reproduce this derivation on a whiteboard.

### Step 1: The RLHF Objective

The standard RLHF objective maximizes expected reward while staying close to the reference policy:

$$\max_{\pi} \mathbb{E}_{x \sim \mathcal{D}, y \sim \pi(\cdot|x)} \left[ r(x, y) \right] - \beta \cdot \text{KL}\left(\pi(\cdot|x) \| \pi_{\text{ref}}(\cdot|x)\right)$$

where:
- $r(x, y)$ is the learned reward model
- $\beta > 0$ controls the KL penalty strength
- $\pi_{\text{ref}}$ is the reference policy (typically the SFT model)

### Step 2: Derive the Optimal Policy

Expanding the KL divergence and writing the objective for a fixed $x$:

$$\max_{\pi} \sum_y \pi(y|x) \left[ r(x,y) - \beta \log \frac{\pi(y|x)}{\pi_{\text{ref}}(y|x)} \right]$$

This is a constrained optimization (since $\pi$ must be a valid distribution). Using the method of Lagrange multipliers (or recognizing this as a KL-regularized optimization with a known solution), the optimal policy is:

$$\boxed{\pi^*(y|x) = \frac{1}{Z(x)} \pi_{\text{ref}}(y|x) \exp\left(\frac{r(x,y)}{\beta}\right)}$$

where $Z(x) = \sum_y \pi_{\text{ref}}(y|x) \exp(r(x,y)/\beta)$ is the partition function.

**Interview tip**: This is a Gibbs/Boltzmann distribution. The reward acts like negative energy, and $\beta$ acts like temperature.

### Step 3: Solve for the Reward

Taking logs of both sides:

$$\log \pi^*(y|x) = \log \pi_{\text{ref}}(y|x) + \frac{r(x,y)}{\beta} - \log Z(x)$$

Rearranging to solve for $r$:

$$\boxed{r(x,y) = \beta \log \frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x)}$$

The key insight: **the reward is fully determined by the log-ratio of optimal policy to reference policy** (plus a prompt-dependent constant that cancels in comparisons).

### Step 4: Substitute into Bradley-Terry Model

The preference model (Bradley-Terry) says:

$$P(y_w \succ y_l | x) = \sigma(r(x, y_w) - r(x, y_l))$$

Substituting our expression for $r$:

$$P(y_w \succ y_l | x) = \sigma\left(\beta \log \frac{\pi^*(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi^*(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)$$

Note that $\beta \log Z(x)$ cancels because it appears in both terms!

### Step 5: The DPO Loss

Now, instead of learning $r$ and then optimizing $\pi$, we directly optimize $\pi$ by maximizing the log-likelihood of the observed preferences:

$$\boxed{\mathcal{L}_{\text{DPO}}(\pi_\theta; \pi_{\text{ref}}) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \left[ \log \sigma\left(\beta \left( \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right)\right)\right]}$$

### Summary of the Derivation

1. Start with RLHF objective (maximize reward - KL penalty)
2. Derive optimal policy (Gibbs distribution)
3. Invert to express reward as function of policy
4. Substitute into Bradley-Terry preference model
5. Partition function $Z(x)$ cancels
6. Train policy directly on preference data via maximum likelihood

**The beauty**: We went from needing to train a reward model + run PPO to just doing supervised learning on preference pairs.

---
## 5. DPO Loss Implementation

In [ ]:
def dpo_loss(
    policy_chosen_logps: torch.Tensor,
    policy_rejected_logps: torch.Tensor,
    ref_chosen_logps: torch.Tensor,
    ref_rejected_logps: torch.Tensor,
    beta: float = 0.1,
    label_smoothing: float = 0.0,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Compute the DPO loss for a batch of preference pairs.
    
    Args:
        policy_chosen_logps: Log probs of chosen responses under policy. Shape: (batch_size,)
        policy_rejected_logps: Log probs of rejected responses under policy. Shape: (batch_size,)
        ref_chosen_logps: Log probs of chosen responses under reference. Shape: (batch_size,)
        ref_rejected_logps: Log probs of rejected responses under reference. Shape: (batch_size,)
        beta: Temperature parameter controlling deviation from reference.
        label_smoothing: Conservative DPO label smoothing (0 = standard DPO).
        
    Returns:
        loss: Scalar DPO loss.
        chosen_rewards: Implicit rewards for chosen responses.
        rejected_rewards: Implicit rewards for rejected responses.
    """
    # Log-ratios: how much policy differs from reference for each response
    chosen_logratios = policy_chosen_logps - ref_chosen_logps
    rejected_logratios = policy_rejected_logps - ref_rejected_logps
    
    # The implicit reward margin (this is what goes into the sigmoid)
    logits = beta * (chosen_logratios - rejected_logratios)
    
    # DPO loss: negative log-sigmoid of the reward margin
    if label_smoothing > 0:
        # Conservative DPO (cDPO): smoothed labels for robustness to noisy preferences
        loss = (
            -F.logsigmoid(logits) * (1 - label_smoothing)
            - F.logsigmoid(-logits) * label_smoothing
        ).mean()
    else:
        loss = -F.logsigmoid(logits).mean()
    
    # Implicit rewards (for monitoring -- not used in loss)
    chosen_rewards = beta * chosen_logratios.detach()
    rejected_rewards = beta * rejected_logratios.detach()
    
    return loss, chosen_rewards, rejected_rewards


print("DPO loss function defined. Let's verify its behavior.")

In [ ]:
def compute_logps(
    model: nn.Module,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    labels: torch.Tensor,
) -> torch.Tensor:
    """
    Compute per-sequence log probabilities for a batch of sequences.
    
    We compute per-token log probs and sum over the response tokens only
    (where labels != -100).
    
    Args:
        model: Causal language model.
        input_ids: Token IDs. Shape: (batch_size, seq_len)
        attention_mask: Attention mask. Shape: (batch_size, seq_len)
        labels: Token IDs for response tokens, -100 for prompt tokens.
                Shape: (batch_size, seq_len)
    
    Returns:
        logps: Sum of log probs over response tokens. Shape: (batch_size,)
    """
    with torch.no_grad() if not model.training else torch.enable_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits  # (batch_size, seq_len, vocab_size)
    
    # Shift logits and labels for next-token prediction
    # logits[:, :-1] predicts labels[:, 1:]
    shift_logits = logits[:, :-1, :]  # (batch_size, seq_len-1, vocab_size)
    shift_labels = labels[:, 1:]       # (batch_size, seq_len-1)
    
    # Compute per-token log probabilities
    log_probs = F.log_softmax(shift_logits, dim=-1)  # (batch_size, seq_len-1, vocab_size)
    
    # Gather log probs for the actual tokens
    per_token_logps = log_probs.gather(
        dim=-1, index=shift_labels.clamp(min=0).unsqueeze(-1)
    ).squeeze(-1)  # (batch_size, seq_len-1)
    
    # Mask out prompt tokens (where labels == -100)
    loss_mask = (shift_labels != -100).float()  # (batch_size, seq_len-1)
    
    # Sum log probs over response tokens
    logps = (per_token_logps * loss_mask).sum(dim=-1)  # (batch_size,)
    
    return logps


print("compute_logps helper defined.")

In [ ]:
# Test DPO loss with synthetic data to verify correct behavior

print("=" * 60)
print("TEST 1: When policy strongly prefers chosen over rejected")
print("=" * 60)

# Policy assigns high probability to chosen, low to rejected
# Reference model is neutral
policy_chosen = torch.tensor([-1.0])   # log P(chosen) under policy
policy_rejected = torch.tensor([-5.0]) # log P(rejected) under policy  
ref_chosen = torch.tensor([-2.0])      # log P(chosen) under reference
ref_rejected = torch.tensor([-2.0])    # log P(rejected) under reference

loss, c_rew, r_rew = dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=0.1)
print(f"Loss: {loss.item():.4f} (should be LOW -- policy is correct)")
print(f"Chosen reward: {c_rew.item():.4f}, Rejected reward: {r_rew.item():.4f}")
print(f"Reward margin: {(c_rew - r_rew).item():.4f} (should be POSITIVE)")

print()
print("=" * 60)
print("TEST 2: When policy wrongly prefers rejected over chosen")
print("=" * 60)

policy_chosen2 = torch.tensor([-5.0])   # policy assigns LOW prob to chosen
policy_rejected2 = torch.tensor([-1.0]) # policy assigns HIGH prob to rejected

loss2, c_rew2, r_rew2 = dpo_loss(policy_chosen2, policy_rejected2, ref_chosen, ref_rejected, beta=0.1)
print(f"Loss: {loss2.item():.4f} (should be HIGH -- policy is wrong)")
print(f"Chosen reward: {c_rew2.item():.4f}, Rejected reward: {r_rew2.item():.4f}")
print(f"Reward margin: {(c_rew2 - r_rew2).item():.4f} (should be NEGATIVE)")

print()
print("=" * 60)
print("TEST 3: Effect of beta")
print("=" * 60)

for beta_val in [0.01, 0.1, 0.5, 1.0, 5.0]:
    l, _, _ = dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=beta_val)
    print(f"beta={beta_val:.2f} -> loss={l.item():.4f}")

print()
print("Observation: Higher beta amplifies the log-ratio difference.")
print("At very high beta, even small policy deviations from reference create large gradients.")
print("At very low beta, the policy can deviate far from reference cheaply.")

### Why does this work?

The DPO loss is doing something elegant: it is simultaneously
1. **Increasing** the probability of chosen responses (relative to reference)
2. **Decreasing** the probability of rejected responses (relative to reference)
3. The **beta** parameter controls how much the policy can deviate from the reference

The gradient of the DPO loss is weighted by how "wrong" the implicit reward model currently is. If the policy already correctly ranks chosen > rejected, the gradient is small. If it ranks them incorrectly, the gradient is large. This is an instance of **implicit reward modeling** -- the policy itself encodes the reward through its log-ratio with the reference.

**Insider Tip:** A subtle point about the DPO derivation -- the partition function Z(x) cancels exactly in the pairwise difference; that step is pure algebra, not an approximation. The real caveats are that DPO's implicit pointwise rewards are only identified up to a prompt-dependent shift, and that the Bradley-Terry assumption is about how preferences relate to rewards, not about whether Z(x) cancels. In practice, DPO can be sensitive to the reference policy. This is a great point to raise in interviews to show depth. See "Is DPO Superior to PPO for LLM Alignment? A Comprehensive Study" (Xu et al. 2024).

**Insider Tip:** Online/iterative DPO significantly outperforms offline DPO. If an interviewer asks "what's the main weakness of DPO?", the answer is "offline data distribution mismatch". The fix is on-policy sampling (generate from current policy, rank, retrain). Snorkel AI's blog and Xiong et al. 2024 ("Iterative Preference Learning from Human Feedback") cover this well.

---
## 6. DPO Trainer from Scratch

In [ ]:
class PreferenceDataset(Dataset):
    """
    Dataset for DPO training. Each item contains:
    - prompt: the input prompt
    - chosen: the preferred response
    - rejected: the dispreferred response
    """
    def __init__(self, data, tokenizer, max_length=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def _tokenize_pair(self, prompt, response):
        """
        Tokenize a prompt+response pair.
        Returns input_ids, attention_mask, and labels (with prompt tokens masked as -100).
        """
        # Tokenize prompt separately to know its length
        prompt_tokens = self.tokenizer(
            prompt, add_special_tokens=True, truncation=True,
            max_length=self.max_length // 2
        )
        prompt_len = len(prompt_tokens['input_ids'])
        
        # Tokenize full sequence
        full_text = prompt + response
        encoded = self.tokenizer(
            full_text, add_special_tokens=True, truncation=True,
            max_length=self.max_length, padding='max_length',
            return_tensors='pt'
        )
        
        input_ids = encoded['input_ids'].squeeze(0)
        attention_mask = encoded['attention_mask'].squeeze(0)
        
        # Create labels: -100 for prompt tokens and padding, actual IDs for response
        # Caveat: BPE merges across the prompt/response boundary can shift the true boundary by a token; production code should tokenize prompt+response jointly and compute the boundary from offsets.
        labels = input_ids.clone()
        labels[:prompt_len] = -100
        labels[attention_mask == 0] = -100
        
        return input_ids, attention_mask, labels
    
    def __getitem__(self, idx):
        item = self.data[idx]
        prompt = item['prompt']
        chosen = item['chosen']
        rejected = item['rejected']
        
        chosen_ids, chosen_mask, chosen_labels = self._tokenize_pair(prompt, chosen)
        rejected_ids, rejected_mask, rejected_labels = self._tokenize_pair(prompt, rejected)
        
        return {
            'chosen_input_ids': chosen_ids,
            'chosen_attention_mask': chosen_mask,
            'chosen_labels': chosen_labels,
            'rejected_input_ids': rejected_ids,
            'rejected_attention_mask': rejected_mask,
            'rejected_labels': rejected_labels,
        }


print("PreferenceDataset defined.")

In [ ]:
class DPOTrainerFromScratch:
    """
    Minimal DPO trainer built from scratch.
    Implements the full DPO training loop with logging.
    """
    
    def __init__(
        self,
        model_name: str = "gpt2",
        beta: float = 0.1,
        lr: float = 1e-5,
        max_length: int = 256,
        label_smoothing: float = 0.0,
    ):
        self.beta = beta
        self.lr = lr
        self.max_length = max_length
        self.label_smoothing = label_smoothing
        
        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        # Load policy model
        print(f"Loading policy model: {model_name}")
        self.policy_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
        
        # Create reference model (frozen deep copy)
        print("Creating reference model (frozen copy)...")
        self.ref_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
        self.ref_model.eval()
        for param in self.ref_model.parameters():
            param.requires_grad = False
        
        # Optimizer
        self.optimizer = torch.optim.AdamW(self.policy_model.parameters(), lr=lr)
        
        # Logging
        self.train_losses = []
        self.reward_margins = []
        self.chosen_rewards_log = []
        self.rejected_rewards_log = []
        
        param_count = sum(p.numel() for p in self.policy_model.parameters())
        print(f"Model parameters: {param_count / 1e6:.1f}M")
        print(f"Beta: {beta}, LR: {lr}, Label smoothing: {label_smoothing}")
        print("Ready to train.")
    
    def _compute_logps_batch(self, model, input_ids, attention_mask, labels):
        """Compute log probabilities for a batch using the model."""
        return compute_logps(model, input_ids, attention_mask, labels)
    
    def train_step(self, batch):
        """
        Single DPO training step.
        
        1. Compute log probs of chosen and rejected under policy
        2. Compute log probs of chosen and rejected under reference (no grad)
        3. Compute DPO loss
        4. Backprop and update
        """
        self.policy_model.train()
        
        # Move batch to device
        chosen_ids = batch['chosen_input_ids'].to(device)
        chosen_mask = batch['chosen_attention_mask'].to(device)
        chosen_labels = batch['chosen_labels'].to(device)
        rejected_ids = batch['rejected_input_ids'].to(device)
        rejected_mask = batch['rejected_attention_mask'].to(device)
        rejected_labels = batch['rejected_labels'].to(device)
        
        # Policy log probs (with gradients)
        policy_chosen_logps = self._compute_logps_batch(
            self.policy_model, chosen_ids, chosen_mask, chosen_labels
        )
        policy_rejected_logps = self._compute_logps_batch(
            self.policy_model, rejected_ids, rejected_mask, rejected_labels
        )
        
        # Reference log probs (no gradients)
        with torch.no_grad():
            ref_chosen_logps = self._compute_logps_batch(
                self.ref_model, chosen_ids, chosen_mask, chosen_labels
            )
            ref_rejected_logps = self._compute_logps_batch(
                self.ref_model, rejected_ids, rejected_mask, rejected_labels
            )
        
        # Compute DPO loss
        loss, chosen_rewards, rejected_rewards = dpo_loss(
            policy_chosen_logps, policy_rejected_logps,
            ref_chosen_logps, ref_rejected_logps,
            beta=self.beta,
            label_smoothing=self.label_smoothing,
        )
        
        # Backprop
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_model.parameters(), max_norm=1.0)
        self.optimizer.step()
        
        # Log metrics
        reward_margin = (chosen_rewards - rejected_rewards).mean().item()
        self.train_losses.append(loss.item())
        self.reward_margins.append(reward_margin)
        self.chosen_rewards_log.append(chosen_rewards.mean().item())
        self.rejected_rewards_log.append(rejected_rewards.mean().item())
        
        return {
            'loss': loss.item(),
            'reward_margin': reward_margin,
            'chosen_reward': chosen_rewards.mean().item(),
            'rejected_reward': rejected_rewards.mean().item(),
        }
    
    def train(self, dataset, batch_size=4, num_epochs=1, log_every=10):
        """Full training loop."""
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        
        total_steps = 0
        for epoch in range(num_epochs):
            print(f"\n--- Epoch {epoch + 1}/{num_epochs} ---")
            for step, batch in enumerate(dataloader):
                metrics = self.train_step(batch)
                total_steps += 1
                
                if total_steps % log_every == 0:
                    print(
                        f"Step {total_steps} | "
                        f"Loss: {metrics['loss']:.4f} | "
                        f"Reward margin: {metrics['reward_margin']:.4f} | "
                        f"Chosen R: {metrics['chosen_reward']:.4f} | "
                        f"Rejected R: {metrics['rejected_reward']:.4f}"
                    )
        
        print(f"\nTraining complete. Total steps: {total_steps}")
        return self.policy_model
    
    def plot_training(self):
        """Plot training metrics."""
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        
        axes[0].plot(self.train_losses)
        axes[0].set_title('DPO Loss')
        axes[0].set_xlabel('Step')
        axes[0].set_ylabel('Loss')
        
        axes[1].plot(self.reward_margins)
        axes[1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
        axes[1].set_title('Implicit Reward Margin (chosen - rejected)')
        axes[1].set_xlabel('Step')
        axes[1].set_ylabel('Margin')
        
        axes[2].plot(self.chosen_rewards_log, label='Chosen', color='green')
        axes[2].plot(self.rejected_rewards_log, label='Rejected', color='red')
        axes[2].set_title('Implicit Rewards Over Training')
        axes[2].set_xlabel('Step')
        axes[2].set_ylabel('Implicit Reward')
        axes[2].legend()
        
        plt.tight_layout()
        plt.show()


print("DPOTrainerFromScratch class defined.")

In [ ]:
# Prepare preference data from Anthropic HH-RLHF
# We'll use a small subset for demonstration

print("Loading Anthropic HH-RLHF dataset...")
raw_dataset = load_dataset("Anthropic/hh-rlhf", split="train", streaming=True)

# Parse the HH-RLHF format: each example has 'chosen' and 'rejected' conversations
def parse_hh_rlhf(example):
    """Parse HH-RLHF format into prompt/chosen/rejected."""
    chosen_text = example['chosen']
    rejected_text = example['rejected']
    
    # The conversations are formatted as:
    # "\n\nHuman: ...\n\nAssistant: ..."
    # Find the last "\n\nAssistant:" to split prompt from response
    
    def split_prompt_response(text):
        # Find the last assistant response
        last_assistant = text.rfind("\n\nAssistant:")
        if last_assistant == -1:
            return text, ""
        prompt = text[:last_assistant + len("\n\nAssistant:")]
        response = text[last_assistant + len("\n\nAssistant:"):]
        return prompt, response
    
    prompt_c, response_c = split_prompt_response(chosen_text)
    prompt_r, response_r = split_prompt_response(rejected_text)
    
    # Use chosen prompt (should be same as rejected prompt)
    return {
        'prompt': prompt_c,
        'chosen': response_c,
        'rejected': response_r,
    }

# Collect a small subset
preference_data = []
for i, example in enumerate(raw_dataset):
    if i >= 500:  # Use 500 examples for demo
        break
    parsed = parse_hh_rlhf(example)
    if len(parsed['chosen']) > 10 and len(parsed['rejected']) > 10:
        preference_data.append(parsed)

print(f"Loaded {len(preference_data)} preference pairs.")
print(f"\nExample prompt (truncated): {preference_data[0]['prompt'][:200]}...")
print(f"Chosen (truncated): {preference_data[0]['chosen'][:100]}...")
print(f"Rejected (truncated): {preference_data[0]['rejected'][:100]}...")

In [ ]:
# Initialize trainer and dataset
trainer = DPOTrainerFromScratch(
    model_name="gpt2",
    beta=0.1,
    lr=1e-5,
    max_length=256,
)

dataset = PreferenceDataset(
    data=preference_data,
    tokenizer=trainer.tokenizer,
    max_length=256,
)

print(f"\nDataset size: {len(dataset)} preference pairs")
print(f"Sample keys: {list(dataset[0].keys())}")

In [ ]:
# Train!
trained_model = trainer.train(
    dataset=dataset,
    batch_size=4,
    num_epochs=1,
    log_every=20,
)

# Plot training curves
trainer.plot_training()

### What to Look For in the Training Plots

1. **Loss should decrease**: The model is learning to assign higher implicit reward to chosen vs. rejected.
2. **Reward margin should increase**: The gap between chosen and rejected implicit rewards should grow, ideally becoming consistently positive.
3. **Chosen rewards should increase, rejected should decrease**: The model learns to upweight preferred responses and downweight dispreferred ones relative to the reference.

If the reward margin plateaus near zero, the model is not learning. If it oscillates wildly, the learning rate or beta may need adjustment.

---
## 7. DPO with TRL

In [ ]:
from trl import DPOTrainer, DPOConfig
from transformers import TrainingArguments
from datasets import Dataset as HFDataset

# TRL's DPOTrainer handles all the boilerplate:
# - Reference model management
# - Log probability computation
# - Loss computation
# - Training loop with proper logging

# Prepare data in TRL format
trl_data = {
    'prompt': [d['prompt'][:512] for d in preference_data[:200]],
    'chosen': [d['chosen'][:256] for d in preference_data[:200]],
    'rejected': [d['rejected'][:256] for d in preference_data[:200]],
}
trl_dataset = HFDataset.from_dict(trl_data)

print(f"TRL dataset size: {len(trl_dataset)}")
print(f"Columns: {trl_dataset.column_names}")

In [ ]:
# Load fresh models for TRL comparison
trl_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
trl_ref_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
trl_tokenizer = AutoTokenizer.from_pretrained("gpt2")
trl_tokenizer.pad_token = trl_tokenizer.eos_token

# Configure DPO training
dpo_config = DPOConfig(
    output_dir="./dpo_trl_output",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    learning_rate=1e-5,
    beta=0.1,  # Same beta as our from-scratch implementation
    max_length=256,
    max_prompt_length=128,
    logging_steps=20,
    save_strategy="no",  # Don't save checkpoints for demo
    remove_unused_columns=False,
    report_to="none",
)

# Initialize TRL DPOTrainer
trl_trainer = DPOTrainer(
    model=trl_model,
    ref_model=trl_ref_model,
    args=dpo_config,
    train_dataset=trl_dataset,
    processing_class=trl_tokenizer,
)

print("TRL DPOTrainer initialized. Starting training...")

In [ ]:
# Train with TRL
trl_trainer.train()

print("\nTRL DPO training complete.")
print("\nCompare this with our from-scratch implementation:")
print("- TRL handles padding, truncation, and batching automatically")
print("- TRL provides wandb/tensorboard integration")
print("- TRL supports PEFT/LoRA out of the box")
print("- Our implementation gives you full control and understanding")

---
## 8. RLHF vs DPO Comparison

### Side-by-Side Generation

In [ ]:
# Generate from DPO model and compare with base model

def generate_response(model, tokenizer, prompt, max_new_tokens=100):
    """Generate a response from a model."""
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

# Test prompts
test_prompts = [
    "\n\nHuman: What is the meaning of life?\n\nAssistant:",
    "\n\nHuman: How do I become a better programmer?\n\nAssistant:",
    "\n\nHuman: Tell me about climate change.\n\nAssistant:",
]

# Load base model for comparison
base_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
base_tokenizer = AutoTokenizer.from_pretrained("gpt2")
base_tokenizer.pad_token = base_tokenizer.eos_token

print("Comparing base GPT-2 vs DPO-trained GPT-2")
print("=" * 70)

for prompt in test_prompts:
    print(f"\nPrompt: {prompt.strip()}")
    print("-" * 50)
    
    base_response = generate_response(base_model, base_tokenizer, prompt)
    dpo_response = generate_response(trained_model, trainer.tokenizer, prompt)
    
    print(f"BASE: {base_response[:200]}")
    print(f"DPO:  {dpo_response[:200]}")
    print()

### Comprehensive Comparison Table

| Dimension | RLHF (PPO) | DPO |
|-----------|-----------|-----|
| **Training phases** | 3 (SFT + RM + RL) | 2 (SFT + DPO) |
| **Models in memory** | 4 (policy + ref + reward + value) | 2 (policy + ref) |
| **GPU memory** | Very high (~4x model size) | Moderate (~2x model size) |
| **Stability** | Notoriously unstable (PPO tuning) | Much more stable |
| **Hyperparameters** | Many (PPO clip, GAE lambda, etc.) | Few (mainly beta) |
| **Data** | Can use online generation | Offline preference pairs only |
| **Reward model** | Explicit, reusable | Implicit (encoded in policy) |
| **Performance ceiling** | Higher (online exploration) | Lower (offline, distribution mismatch) |
| **Implementation** | Complex (PPO, advantage estimation) | Simple (supervised loss) |
| **Training time** | Longer (RL loop is slow) | Shorter (standard batch SGD) |
| **Scalability** | Harder to scale | Easier to scale |

### Key Takeaway

DPO trades **theoretical optimality** for **practical simplicity**. For most applications, DPO is the better starting point. RLHF may still win when:
- You need online data collection and exploration
- Your preference data is limited and you want to generate more
- You are pushing for maximum performance and can invest in engineering

---
## 9. When DPO Fails

Understanding failure modes is crucial for interviews. Here are the main ways DPO can go wrong:

### 9.1 Offline Data Staleness (Distribution Mismatch)

DPO trains on a **fixed** dataset of preference pairs. As the policy changes during training, the responses it would generate diverge from the responses in the training data. This creates a **distribution mismatch**: the policy is being optimized on data that no longer represents its own behavior.

- **RLHF doesn't have this problem** because PPO generates fresh responses at each step (online RL)
- **Mitigation**: Iterative DPO (regenerate responses periodically), or online DPO variants

### 9.2 Sensitivity to Beta

The beta parameter controls the KL penalty:

| Beta Value | Behavior |
|-----------|----------|
| Too high (e.g., 1.0+) | Policy barely moves from reference. Underfitting preferences. |
| Too low (e.g., 0.01) | Policy diverges far from reference. Overfits to noisy preferences. May degenerate. |
| Sweet spot (0.1-0.5) | Typically good, but depends on data quality and model size. |

### 9.3 Preference Data Quality

**Garbage in, garbage out.** Common issues:
- **Noisy labels**: Human annotators disagree; some "chosen" responses are actually worse
- **Annotation artifacts**: Length bias (longer responses often preferred), formatting bias
- **Narrow coverage**: Data only covers certain topics/styles, policy doesn't generalize

### 9.4 Length Exploitation

DPO models frequently learn to **generate longer responses** because:
1. Human annotators often prefer longer, more detailed responses
2. The sum of log-probabilities (unnormalized) scales with length
3. The model can "game" the implicit reward by being verbose

**Mitigations**:
- Length-normalized log probabilities (this is what SimPO does -- see next notebook)
- Adding a length penalty to the loss
- Filtering training data for length balance

### 9.5 Mode Collapse

Without sufficient KL penalty (beta too low), the policy can collapse to producing a narrow set of responses that happen to score well on the training preferences, losing diversity.

In [ ]:
# Demonstrating beta sensitivity

# Simulate a scenario: policy has mild preference for chosen
policy_chosen_lp = torch.tensor([-3.0])
policy_rejected_lp = torch.tensor([-4.0])
ref_chosen_lp = torch.tensor([-3.5])
ref_rejected_lp = torch.tensor([-3.5])

betas = np.linspace(0.01, 2.0, 50)
losses = []
margins = []

for b in betas:
    l, cr, rr = dpo_loss(policy_chosen_lp, policy_rejected_lp,
                         ref_chosen_lp, ref_rejected_lp, beta=b)
    losses.append(l.item())
    margins.append((cr - rr).item())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(betas, losses)
ax1.set_xlabel('Beta')
ax1.set_ylabel('DPO Loss')
ax1.set_title('Loss vs Beta (fixed log-probs)')
ax1.axvline(x=0.1, color='r', linestyle='--', alpha=0.5, label='beta=0.1')
ax1.legend()

ax2.plot(betas, margins)
ax2.set_xlabel('Beta')
ax2.set_ylabel('Reward Margin')
ax2.set_title('Implicit Reward Margin vs Beta')
ax2.axvline(x=0.1, color='r', linestyle='--', alpha=0.5, label='beta=0.1')
ax2.legend()

plt.tight_layout()
plt.show()

print("At low beta: loss is very low (saturated sigmoid), small gradients, policy can diverge freely.")
print("At high beta: loss is high even for correct ranking, large gradients, policy is constrained.")

---
## 10. "Why Does This Work?" -- Deep Understanding Prompts

These are the questions an interviewer at a frontier lab would ask. Make sure you can answer each one.

---

### Q: "Why is the reference model needed?"

**Answer**: The reference model serves as an anchor that prevents the policy from degenerating. Without it, the policy could learn to assign all probability mass to a single "chosen" response for each prompt, collapsing to a deterministic, low-diversity model. The KL penalty (encoded via the log-ratio $\pi/\pi_{\text{ref}}$) forces the policy to stay close to a well-behaved language model while shifting probability toward preferred responses.

More formally, without the reference, the optimal solution to "maximize probability of chosen, minimize probability of rejected" would be a degenerate distribution. The reference term regularizes this into a smooth policy.

---

### Q: "What happens if you remove the reference model?"

**Answer**: This is exactly what SimPO does (Meng et al., 2024). Instead of using a reference model, SimPO uses the **average log-likelihood** of the response as an implicit reward, with a target reward margin gamma. This eliminates the need for a reference model. Note that the frozen reference is forward-only (no gradients or optimizer states), so removing it saves well under 50% of training memory -- the bigger savings are compute and engineering simplicity. However, it requires careful tuning of the length normalization and margin parameters.

We will implement SimPO in the next notebook (08-alignment-variants).

---

### Q: "Is DPO strictly better than RLHF?"

**Answer**: No. DPO makes a strong assumption: the preference data is representative of the optimal policy's behavior. In practice:

1. **Online vs. Offline**: RLHF with PPO is an *online* algorithm that generates fresh data, while DPO is *offline* (fixed dataset). Online algorithms can explore and find better responses than what's in the training data.

2. **Empirical results**: Papers like "Is DPO Superior to PPO for LLM Alignment?" (Xu et al., 2024) show that PPO can outperform DPO when both are well-tuned, especially on complex reasoning tasks.

3. **The practical reality**: DPO is easier to implement, more stable, and good enough for most use cases. RLHF may squeeze out more performance at the frontier.

---

### Q: "What is the gradient of the DPO loss? Intuitively, what does it do?"

**Answer**: The gradient of the DPO loss has the form:

$$\nabla_\theta \mathcal{L} = -\beta \cdot \sigma(-\hat{r}) \cdot \left[\nabla_\theta \log \pi_\theta(y_w|x) - \nabla_\theta \log \pi_\theta(y_l|x)\right]$$

where $\hat{r} = \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}$ is the implicit reward margin.

**Intuition**: The weighting term $\sigma(-\hat{r})$ is high when the implicit reward model is wrong (chosen has lower implicit reward than rejected) and low when it is correct. So the gradient focuses on examples where the model is making mistakes, which is an efficient use of gradient updates.

---
## Interview Question Bank: DPO

*A representative set of senior-level interview questions for post-training and alignment roles. These are the questions that separate "read the paper" from "could lead this work."*

---

**Q1: "Derive the DPO loss from the RLHF objective on the whiteboard."**

**What we're testing:** Mathematical maturity, first-principles thinking, ability to communicate technical ideas clearly under pressure.

**Context:** This is a representative senior-level interview question. You will get 20-30 minutes. You MUST be able to do this cold -- no notes, no prompts. If you cannot, you are not ready for a Senior ML researcher role in post-training.

**Good answer:**
- Starts from the KL-constrained reward maximization objective: max E[r(x,y)] - beta * KL(pi || pi_ref)
- Derives the closed-form optimal policy: pi*(y|x) = pi_ref(y|x) * exp(r(x,y)/beta) / Z(x)
- Rearranges to express r(x,y) in terms of policies: r(x,y) = beta * log(pi*(y|x) / pi_ref(y|x)) + beta * log Z(x)
- Substitutes into Bradley-Terry preference model: p(y_w > y_l) = sigma(r(y_w) - r(y_l))
- Shows Z(x) cancels in the difference, arriving at: L_DPO = -log sigma(beta * (log(pi/pi_ref)_w - log(pi/pi_ref)_l))
- Gets to the correct final loss.

**Great answer (Senior -> Principal level):**
- Explains the *intuition* at each step: "We are exploiting the fact that the optimal policy has a known analytical form under KL constraints -- this is the key insight that makes RLHF without RL possible."
- Discusses what assumptions are made and when they break:
  - Bradley-Terry model assumes preferences are transitive and context-independent (real human preferences are neither)
  - The partition function Z(x) cancels only in the pairwise case -- this is why DPO needs paired preferences
  - The derivation assumes the reward model is accurate, but we never explicitly train one
- Notes the subtle issue: DPO implicitly defines a reward model, but that implicit reward can be degenerate (the Azar et al. IPO critique)
- Connects to the broader landscape: "This same rearrangement trick is what enables the entire family of direct alignment methods -- KTO, SimPO, etc. each modify a different assumption in this derivation."

**Red flag:** Cannot get past step 2. Memorized the final loss but cannot explain why Z(x) cancels. Says "it is just cross-entropy" without deeper understanding.

**Follow-up:** "The DPO loss assumes the Bradley-Terry model. What if preferences are not transitive? For example, annotator says A > B, B > C, but C > A."
- Expected: Discuss limitations of Bradley-Terry, mention Plackett-Luce or Thurstone models, note that real human preferences are noisy and potentially cyclic. A great candidate mentions Nash-MD or Self-Play Preference Optimization (SPPO) as alternatives designed for non-transitive preferences.

**Follow-up:** "Implement DPO loss in PyTorch in 10 lines."
- Expected (clean version):
```python
def dpo_loss(pi_logps_w, pi_logps_l, ref_logps_w, ref_logps_l, beta=0.1):
    pi_logratios = pi_logps_w - pi_logps_l
    ref_logratios = ref_logps_w - ref_logps_l
    logits = beta * (pi_logratios - ref_logratios)
    return -F.logsigmoid(logits).mean()
```

---

**Q2: "You have trained a DPO model and it is producing longer but not better responses. Diagnose and fix."**

**What we're testing:** Debugging intuition, production experience, knowledge of failure modes beyond the textbook.

**Good answer:**
- Identifies the core problem: length exploitation / verbosity bias
- The preference data likely has a length bias (longer responses were preferred by annotators)
- DPO learns to increase log-probability of longer sequences, which naturally have lower per-token log-prob, creating an implicit length reward
- Fix 1: Increase beta to strengthen the reference model constraint
- Fix 2: Add length penalty or length normalization to the loss

**Great answer (Senior -> Principal level):**
- Discusses the SimPO approach: length-normalized log probabilities as the implicit reward, which directly addresses this failure mode
- Mentions reference model strength: if the reference model is weak (e.g., base model vs SFT model), the KL constraint is less meaningful
- Suggests a data quality audit: "Before changing the algorithm, check if the preference data itself has a length bias. If annotators consistently preferred longer responses, the model is learning the correct signal from bad data."
- Proposes switching to online DPO: generate on-policy data, have it re-ranked, which naturally corrects distribution drift
- Knows about DPO's implicit reward model and can diagnose whether it has collapsed to a length proxy using reward model probing

**Red flag:** Only suggests "reduce learning rate" or "train longer." Does not mention length exploitation at all. Shows no understanding of the implicit reward model.

**Follow-up:** "How would you detect length exploitation automatically in a training pipeline before it becomes a problem?"
- Expected: Monitor correlation between response length and implicit reward, track length distribution over training, compare win rates on length-controlled subsets.

---

**Q3: "DPO vs PPO -- give me a nuanced comparison, not just 'DPO is simpler.'"**

**What we're testing:** Depth of understanding, ability to reason about trade-offs at a systems level, awareness of when each method is appropriate.

**Good answer:**
- Offline vs online: DPO is offline (fixed preference dataset), PPO is online (generates new data during training)
- Compute: DPO needs only the policy and reference model. PPO needs policy, reference, reward model, and value model (4 models in memory).
- Stability: DPO is more stable (supervised loss), PPO requires careful hyperparameter tuning (clip range, GAE lambda, entropy bonus, learning rate schedules)
- Data requirements: DPO needs paired preferences, PPO needs a reward model trained on preferences

**Great answer (Senior -> Principal level):**
- "PPO can explore beyond the training data distribution. DPO is fundamentally bounded by the preference data it was trained on. For novel capabilities -- especially reasoning, where the model needs to discover new solution strategies -- PPO (or GRPO) wins because it can explore. For style alignment or safety, where we want to stay close to known-good behavior, DPO is sufficient and more efficient."
- Discusses the online DPO variants (iterative DPO, online DPO with rejection sampling) as a middle ground that partially addresses the exploration limitation
- Notes that in practice, frontier labs use both: DPO for initial alignment, then PPO/GRPO for capability-specific training (especially reasoning)
- Can quantify: "PPO costs roughly 4x the memory and 2-3x the compute per step, but the gains in out-of-distribution performance can be worth it for capability-critical applications"
- Mentions that DeepSeek-R1's success with GRPO (a PPO variant) for reasoning is strong evidence that online RL exploration matters for capability elicitation

**Red flag:** "DPO is strictly better because it is simpler." Shows no understanding of the exploration-exploitation trade-off. Cannot explain when you would choose PPO over DPO despite the added complexity.

---
## Production Implementation Notes: DPO

These are the things that matter when you move DPO from a research notebook to a training cluster. Interviewers at frontier labs will probe whether you have shipped alignment at scale or only read about it.

### DPO at Scale: Distributed Training

- **Reference model sharding**: The reference model is frozen, so shard it separately from the policy model. In practice, the reference model lives on its own set of GPUs with FSDP/DeepSpeed ZeRO-3, and you only call it for forward passes. This halves the memory pressure on the policy model's GPUs.
- **Gradient checkpointing**: Essential. DPO requires forward passes through both policy and reference models for both chosen and rejected sequences -- that is 4 forward passes per step. Without gradient checkpointing, you will OOM on anything above 7B parameters.
- **Mixed precision**: BF16 for training, FP32 for loss computation. The log-sigmoid in the DPO loss can underflow in FP16.
- **Batch size**: Larger batches stabilize DPO significantly. Effective batch sizes of 64-256 preference pairs are standard. Use gradient accumulation if needed.

### Iterative DPO (How It Actually Works at Frontier Labs)

The single-shot "train DPO once" recipe from the paper is rarely used in production. The standard practice:

1. **Round 1**: Train DPO on initial preference dataset
2. **Generate on-policy data**: Sample from the DPO-trained model on a diverse prompt set
3. **Re-rank or re-label**: Use a reward model (or human annotators) to create new preference pairs from the on-policy generations
4. **Round 2**: Train DPO again, using the new on-policy preference data (and optionally mixing in old data)
5. **Repeat**: 2-6 iterations is typical. Diminishing returns after 3-4.

This is the "online DPO" or "iterative DPO" approach described in the Llama 3 paper (Section on RLHF). It partially addresses DPO's core limitation (offline, fixed data) without the full complexity of PPO.

### Evaluation Pipeline

- **Win rate vs reference**: Generate from both DPO model and reference (SFT) model on 500-1000 diverse prompts. Use a strong reward model or GPT-4 as judge to compute win rate. Target: 60-70% win rate.
- **Safety regression testing**: Run the model through a fixed safety benchmark (e.g., harmbench, red-team prompts). DPO can regress on safety if the preference data does not include safety-relevant examples. Always test.
- **Human eval on diverse prompt set**: Automated metrics lie. Budget for 200-500 human A/B comparisons before shipping. Focus on edge cases: long-form, multi-turn, instruction-following, refusals.
- **Length distribution monitoring**: Plot response length distributions before and after DPO. If the mean length increases by more than 20%, investigate length exploitation.
- **Perplexity on held-out data**: Ensure the model has not diverged too far from the reference. A large perplexity increase signals overfitting to the preference data.

---
## How DPO Gets Tested in Interviews

### Interview Format Expectations

Formats vary by lab, team, and point in time, so do not over-index on any one rubric. Expect some mix of derivation on a whiteboard, coding, and open-ended discussion of trade-offs and failure modes; verify the specifics of your loop with your recruiter.

### The 4-Week Prep Plan for DPO Questions

**Week 1: Foundation**
- Derive DPO loss from scratch 3 times on paper (no notes). Time yourself. Target: under 15 minutes.
- Implement DPO loss in PyTorch from memory. Run it on synthetic data.
- Read the DPO paper (Rafailov et al. 2023) Section 4 carefully. Understand every step.

**Week 2: Failure Modes**
- Implement and reproduce the length exploitation failure. Show it, then fix it with SimPO-style normalization.
- Train DPO with different beta values. Build intuition for what beta controls.
- Read the IPO paper (Azar et al. 2023) -- it is the most cited critique of DPO. Know the "implicit reward is degenerate" argument.

**Week 3: Production**
- Set up distributed DPO training with TRL + DeepSpeed. Know the config options.
- Implement iterative DPO: train, generate, re-rank, retrain. This is how it actually works.
- Read Llama 3 paper Section 4.2 (RLHF details). This is the industry-standard recipe.

**Week 4: Mock Interviews**
- Have someone ask you Q1-Q3 above. Record yourself. Watch for: do you explain intuition or just recite steps? Do you connect to the broader landscape?
- Practice transitioning from "here is the math" to "here is how I would use this in practice." That transition is what separates Senior from Principal.

### Signals That Differentiate Levels

| Signal | Senior | Principal |
|--------|--------|-----------|
| Derivation | Gets to correct loss | Explains *why* each step works and what breaks if assumptions change |
| Failure modes | Names 2-3 known issues | Has debugged them in practice, can describe the fix AND the detection mechanism |
| System design | "Use DPO with good data" | "Here is my 6-round iterative pipeline with safety gates, human eval checkpoints, and automated length monitoring" |
| Breadth | Knows DPO vs PPO | Knows DPO vs PPO vs GRPO vs KTO vs SimPO, and can build a decision tree for when to use each |
| Research taste | "DPO is better than RLHF" | "DPO solved the stability problem but created the exploration problem. The field is converging on online DPO + GRPO for different use cases." |

---
## 11. Flashcard Summary (Anki-Ready Q&A)

| # | Question | Answer |
|---|----------|--------|
| 1 | What is DPO? | Direct Preference Optimization. A method that trains a policy directly on preference data without learning an explicit reward model, by exploiting the closed-form solution of the RLHF objective. |
| 2 | How many models does DPO need in memory? | 2: the policy model (being trained) and the reference model (frozen). Compare with RLHF's 4 (policy + ref + reward + value). |
| 3 | What is the RLHF objective that DPO starts from? | max E[r(x,y)] - beta * KL(pi \|\| pi_ref). Maximize reward while staying close to reference policy. |
| 4 | What is the optimal policy under the RLHF objective? | pi*(y\|x) = (1/Z(x)) * pi_ref(y\|x) * exp(r(x,y)/beta). A Gibbs/Boltzmann distribution. |
| 5 | How do you express reward in terms of policy? | r(x,y) = beta * log(pi*(y\|x) / pi_ref(y\|x)) + beta * log(Z(x)). |
| 6 | Why does Z(x) cancel in the DPO loss? | Because the Bradley-Terry model uses r(y_w) - r(y_l), and Z(x) is the same for both (it only depends on the prompt x). |
| 7 | Write the DPO loss. | L = -E[log sigma(beta * (log(pi(y_w\|x)/pi_ref(y_w\|x)) - log(pi(y_l\|x)/pi_ref(y_l\|x))))] |
| 8 | What does beta control? | The strength of the KL penalty. High beta = policy stays close to reference (conservative). Low beta = policy can deviate far (aggressive). |
| 9 | What is the reference model? | A frozen copy of the SFT model that anchors the policy, preventing degeneration and mode collapse. |
| 10 | What is the implicit reward in DPO? | beta * log(pi(y\|x) / pi_ref(y\|x)). How much the policy upweights a response relative to reference. |
| 11 | What is the main failure mode of DPO? | Offline data staleness: since DPO trains on fixed data, there is distribution mismatch as the policy changes. Online RL methods don't have this issue. |
| 12 | How does DPO handle length exploitation? | Standard DPO does not -- it uses unnormalized log-probs that scale with length. SimPO fixes this with length normalization. |
| 13 | What is conservative DPO (cDPO)? | DPO with label smoothing: L = (1-eps)*(-log sigma(logits)) + eps*(-log sigma(-logits)). Robust to noisy preference labels. |
| 14 | What does the DPO gradient weight sigma(-r_hat) mean? | The gradient focuses on examples where the implicit reward model is wrong. When the model correctly ranks chosen > rejected, the gradient is small (efficient learning). |
| 15 | Is DPO always better than RLHF? | No. Online RL (PPO) can outperform offline DPO, especially on complex reasoning tasks, because it can explore and generate fresh data. DPO wins on simplicity and stability. |

---
## 12. Paper Guide: DPO (Rafailov et al., 2023)

**Paper**: "Direct Preference Optimization: Your Language Model is Secretly a Reward Model"  
**Link**: [https://arxiv.org/abs/2305.18290](https://arxiv.org/abs/2305.18290)

### Section-by-Section Reading Guide

| Section | What to Focus On | Time |
|---------|-----------------|------|
| **Abstract + Intro (Sec 1)** | The motivation: RLHF is complex and unstable. DPO is a simpler alternative. Note the key claim: DPO is as effective as RLHF. | 10 min |
| **Preliminaries (Sec 2)** | The RLHF pipeline review. Make sure you understand the 3 phases. The Bradley-Terry model for preferences. | 10 min |
| **DPO Derivation (Sec 3)** | **This is the most important section.** Follow the derivation step by step. Understand each equation. This is what you implemented above. Pay special attention to the partition function cancellation. | 30 min |
| **Sec 3.2 - DPO Update** | The gradient analysis. The weighting by sigma(-r_hat) is the key insight about efficient gradient updates. | 15 min |
| **Theoretical Analysis (Sec 4)** | Skim. Shows DPO optimizes the same objective as RLHF under certain conditions. Useful for "is DPO equivalent to RLHF?" interview questions. | 10 min |
| **Experiments (Sec 5)** | DPO vs. PPO on summarization (TL;DR), dialogue (Anthropic HH), and controlled generation. Note: DPO matches or exceeds PPO with much less compute. | 15 min |
| **Sec 5.2 - IMDb controlled generation** | Clean, small-scale experiment that clearly shows DPO works. Good for building intuition. | 5 min |
| **Discussion (Sec 6)** | Limitations and future work. The offline data issue is acknowledged. | 5 min |

### Key Equations to Memorize

1. **Optimal policy**: $\pi^*(y|x) = \frac{1}{Z(x)} \pi_{\text{ref}}(y|x) \exp(r(x,y)/\beta)$ (Eq. 4)
2. **Reward from policy**: $r(x,y) = \beta \log \frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x)$ (Eq. 5)
3. **DPO loss**: Eq. 7 -- the final loss function
4. **DPO gradient**: Eq. 8 -- weighted by how wrong the implicit reward is

### Interview Tips from This Paper

- Be prepared to derive the DPO loss from scratch (the most common alignment interview question)
- Know the connection between DPO and reward modeling (the policy IS the reward model)
- Understand why Z(x) cancels (partition function is prompt-dependent, cancels in pairwise comparison)
- Be able to explain the gradient weighting (sigma(-r_hat)) intuitively
- Know the limitations: offline data, beta sensitivity, length exploitation

### Related Papers to Skim

- **"Is DPO Superior to PPO for LLM Alignment?"** (Xu et al., 2024) -- careful comparison showing PPO can win
- **"Statistical Rejection Sampling"** (Liu et al., 2023) -- alternative to PPO for RLHF
- **KTO, IPO, SimPO, ORPO** -- variants covered in the next notebook